# Tile + Predict → Label Studio Import (Dockerized)

Processes all images in `data/raw/big/`:

1. Adaptively estimates stitch size per image
2. Cuts each image into tiles at the effective tile size
3. Runs YOLO OBB prediction on every tile
4. Saves each tile **inside `mydata/big_tiles/`** (Label Studio's Docker
   container has `mydata` bind-mounted to `/label-studio/data`, so it can
   serve these files directly)
5. Saves each tile's detections as YOLO OBB labels
6. Builds `tasks.json` with `/data/local-files/?d=big_tiles/...` URLs

**Output structure (all inside `mydata/` so LS Docker can read it):**
```
mydata/big_tiles/
├── classes.txt
├── tasks.json              ← Import this into Label Studio
├── images/*.jpg
└── labels/*.txt            (raw YOLO OBB, normalized per tile)
```

**Docker setup (the one you already use):**
```
docker run -it -p 8080:8080 \
  -v $(pwd)/mydata:/label-studio/data \
  -e LOCAL_FILES_SERVING_ENABLED=true \
  -e LOCAL_FILES_DOCUMENT_ROOT=/label-studio/data \
  heartexlabs/label-studio:latest
```

With that mount, the URL `/data/local-files/?d=big_tiles/3_tile_000.jpg` in
`tasks.json` automatically resolves to `mydata/big_tiles/3_tile_000.jpg`
on disk.

If your `mydata` folder lives somewhere else (not at the project root),
edit `MYDATA_DIR` in cell 1.


In [7]:
import os
import sys
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

PROJECT_DIR = os.path.abspath(".")
sys.path.insert(0, PROJECT_DIR)

INPUT_DIR   = os.path.join(PROJECT_DIR, "data", "raw", "big")

# Label Studio Docker setup: the `mydata` folder is bind-mounted into the
# container at /label-studio/data, so anything placed inside mydata/<subdir>/
# is served at /data/local-files/?d=<subdir>/<filename>.
#
# Typical Docker run:
#     docker run -it -p 8080:8080 \
#         -v $(pwd)/mydata:/label-studio/data \
#         -e LOCAL_FILES_SERVING_ENABLED=true \
#         -e LOCAL_FILES_DOCUMENT_ROOT=/label-studio/data \
#         heartexlabs/label-studio:latest
MYDATA_DIR      = os.path.join(PROJECT_DIR, "mydata", "test_data")   # host-side folder mounted into LS container
TILE_SUBDIR     = "big_tiles"                           # subfolder inside mydata
OUTPUT_DIR      = os.path.join(MYDATA_DIR, TILE_SUBDIR)

# URL prefix that Label Studio uses to serve files out of LOCAL_FILES_DOCUMENT_ROOT.
# With LOCAL_FILES_DOCUMENT_ROOT=/label-studio/data (which maps to host mydata/),
# a file at mydata/big_tiles/foo.jpg is served at:
LS_URL_PREFIX   = f"/data/local-files/?d=test_data/{TILE_SUBDIR}/images/"

MODEL_PATH  = os.path.join(PROJECT_DIR, "runs", "obb", "train21", "weights", "best.pt")

# YOLO / tiling parameters
TILE_SIZE        = 640
TARGET_STITCH_PX = 100
OVERLAP          = 0.25
CONF             = 0.20
IOU_NMS          = 0.45
MIN_TILE         = 128

# 9-class system matching training_data/data.yaml
CLASS_NAMES = [
    "chain",         # 0
    "double",        # 1
    "double treble", # 2
    "enseble_chain", # 3
    "fan",           # 4
    "half_double",   # 5
    "noise",         # 6
    "single",        # 7
    "treble",        # 8
]

# Prepare output directories (inside mydata/ so Label Studio can see them)
out_root = Path(OUTPUT_DIR)
out_root.mkdir(parents=True, exist_ok=True)
(out_root / "images").mkdir(exist_ok=True)
(out_root / "labels").mkdir(exist_ok=True)

# Write classes.txt
with open(out_root / "classes.txt", "w") as f:
    for name in CLASS_NAMES:
        f.write(name + "\n")

print(f"Input dir:        {INPUT_DIR}")
print(f"Output dir:       {OUTPUT_DIR}")
print(f"   (inside mydata/ so Label Studio Docker can serve the files)")
print(f"LS URL prefix:    {LS_URL_PREFIX}")
print(f"Model:            {MODEL_PATH}")
print(f"Tile:             {TILE_SIZE} (target stitch px: {TARGET_STITCH_PX}, overlap: {OVERLAP})")
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")


Input dir:        /Users/elevchenko/Documents/DataScience/Crochet/data/raw/big
Output dir:       /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles
   (inside mydata/ so Label Studio Docker can serve the files)
LS URL prefix:    /data/local-files/?d=test_data/big_tiles/images/
Model:            /Users/elevchenko/Documents/DataScience/Crochet/runs/obb/train21/weights/best.pt
Tile:             640 (target stitch px: 100, overlap: 0.25)
Classes (9): ['chain', 'double', 'double treble', 'enseble_chain', 'fan', 'half_double', 'noise', 'single', 'treble']


In [8]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO
from util.tiler import estimate_stitch_size, _tile_starts

# Load model
print(f"Loading model from {MODEL_PATH}")
model = YOLO(MODEL_PATH)
print("Model loaded.")


Loading model from /Users/elevchenko/Documents/DataScience/Crochet/runs/obb/train21/weights/best.pt
Model loaded.


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# Core function: tile one image + save each tile with YOLO OBB predictions
# ═══════════════════════════════════════════════════════════════════════════════

def tile_and_save(image_path, model, out_root, tile_idx_start=0):
    """
    Tile `image_path` using the same adaptive strategy as predict_adaptive,
    run YOLO on each tile, and save tile + YOLO-OBB label file.

    Returns total number of tiles written and total detections.
    """
    image_path = Path(image_path)
    image = cv.imread(str(image_path))
    if image is None:
        print(f"  ✗ Could not read: {image_path}")
        return 0, 0

    h, w = image.shape[:2]
    stem = image_path.stem.replace(" ", "_").replace("(", "").replace(")", "")

    # ── 1. Estimate stitch size ────────────────────────────────────────────
    est = estimate_stitch_size(model, image, tile_size=TILE_SIZE, conf=0.15)
    if est is None or est <= 0:
        print(f"  [{stem}] No stitches detected in size-estimation pass; using full TILE_SIZE")
        effective_tile = min(TILE_SIZE, min(h, w))
    else:
        effective_tile = int(TILE_SIZE * est / TARGET_STITCH_PX)
        effective_tile = max(effective_tile, MIN_TILE)
        effective_tile = min(effective_tile, min(h, w))  # cannot exceed image

    # If image is smaller than one tile, treat it as a single tile
    if effective_tile >= max(h, w):
        x_starts, y_starts = [0], [0]
        eff_w, eff_h = w, h
    else:
        stride = max(1, int(effective_tile * (1 - OVERLAP)))
        y_starts = _tile_starts(h, effective_tile, stride)
        x_starts = _tile_starts(w, effective_tile, stride)
        eff_w = eff_h = effective_tile

    est_str = f"{est:.1f}" if est else "n/a"
    print(f"  [{stem}] size={w}×{h}  est_stitch={est_str}  "
          f"effective_tile={effective_tile}  tiles={len(y_starts)}×{len(x_starts)}")

    total_tiles = 0
    total_detections = 0

    # ── 2. Process each tile ───────────────────────────────────────────────
    for yi, y1 in enumerate(y_starts):
        for xi, x1 in enumerate(x_starts):
            y2 = min(y1 + eff_h, h)
            x2 = min(x1 + eff_w, w)
            tile = image[y1:y2, x1:x2]
            th, tw = tile.shape[:2]

            # Run YOLO on this tile (native resolution; YOLO handles its own resize)
            res = model.predict(tile, conf=CONF, iou=IOU_NMS, verbose=False)[0]

            # Build label lines (normalized to the tile)
            lines = []
            for box in res.obb:
                cls_id = int(box.cls[0])
                corners = box.xyxyxyxy.cpu().numpy().reshape(4, 2).astype(np.float32)
                # Clip and normalize
                coords = []
                for (cx, cy) in corners:
                    cx_n = float(np.clip(cx / tw, 0.0, 1.0))
                    cy_n = float(np.clip(cy / th, 0.0, 1.0))
                    coords.append(cx_n)
                    coords.append(cy_n)
                lines.append(f"{cls_id} " + " ".join(f"{c:.6f}" for c in coords))

            tile_idx = tile_idx_start + total_tiles
            tile_name = f"{stem}_tile_{tile_idx:03d}"
            img_out = out_root / "images" / f"{tile_name}.jpg"
            lbl_out = out_root / "labels" / f"{tile_name}.txt"

            cv.imwrite(str(img_out), tile, [cv.IMWRITE_JPEG_QUALITY, 92])
            with open(lbl_out, "w") as f:
                f.write("\n".join(lines))
                if lines:
                    f.write("\n")

            total_tiles += 1
            total_detections += len(lines)

    print(f"  [{stem}] → {total_tiles} tiles, {total_detections} detections")
    return total_tiles, total_detections

print("tile_and_save() ready.")


tile_and_save() ready.


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# Process every image in data/raw/big
# ═══════════════════════════════════════════════════════════════════════════════

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

input_path = Path(INPUT_DIR)
image_files = sorted(p for p in input_path.iterdir()
                     if p.is_file() and p.suffix.lower() in IMG_EXTS)

print(f"Found {len(image_files)} images in {INPUT_DIR}")

grand_tiles = 0
grand_dets  = 0

for img_path in image_files:
    print(f"\n▶ Processing {img_path.name}")
    n_tiles, n_dets = tile_and_save(img_path, model, out_root,
                                    tile_idx_start=0)
    grand_tiles += n_tiles
    grand_dets  += n_dets

print("\n" + "=" * 60)
print(f"DONE: {grand_tiles} tiles, {grand_dets} detections total")
print(f"Images → {out_root / 'images'}")
print(f"Labels → {out_root / 'labels'}")
print(f"Classes → {out_root / 'classes.txt'}")


Found 1 images in /Users/elevchenko/Documents/DataScience/Crochet/data/raw/big

▶ Processing 11.jpg
  [11] size=461×1001  est_stitch=20.2  effective_tile=129  tiles=11×5
  [11] → 55 tiles, 10498 detections

DONE: 55 tiles, 10498 detections total
Images → /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/images
Labels → /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/labels
Classes → /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/classes.txt


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# Quick visual QA: render a grid of random tiles with their predicted OBBs
# ═══════════════════════════════════════════════════════════════════════════════

import random
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sample_count = 8
all_tiles = sorted((out_root / "images").glob("*.jpg"))
samples = random.sample(all_tiles, min(sample_count, len(all_tiles))) if all_tiles else []

# Color per class
cmap = plt.cm.tab10(np.linspace(0, 1, len(CLASS_NAMES)))

if samples:
    cols = 4
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = np.array(axes).reshape(-1)

    for ax, tile_path in zip(axes, samples):
        img = cv.imread(str(tile_path))
        img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        th, tw = img.shape[:2]
        ax.imshow(img_rgb)

        lbl_path = out_root / "labels" / (tile_path.stem + ".txt")
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 9:
                        continue
                    cls_id = int(parts[0])
                    coords = list(map(float, parts[1:9]))
                    xs = [coords[2*i] * tw for i in range(4)]
                    ys = [coords[2*i + 1] * th for i in range(4)]
                    xs.append(xs[0]); ys.append(ys[0])
                    color = cmap[cls_id % len(cmap)]
                    ax.plot(xs, ys, color=color, linewidth=1.5)
                    ax.text(xs[0], ys[0] - 3, CLASS_NAMES[cls_id],
                            color='white', fontsize=6, fontweight='bold',
                            bbox=dict(facecolor=color, edgecolor='none',
                                      alpha=0.8, pad=1))
        ax.set_title(tile_path.stem, fontsize=8)
        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.tight_layout()
    qa_path = out_root / "_qa_preview.png"
    plt.savefig(qa_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"QA preview saved to: {qa_path}")
else:
    print("No tiles to preview.")


QA preview saved to: /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/_qa_preview.png


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# Build tasks.json directly (Label Studio format, Docker-compatible URLs)
# ═══════════════════════════════════════════════════════════════════════════════
# Generates a Label Studio tasks.json with OBB pre-annotations using
# RectangleLabels + rotation (the standard LS oriented bounding box format).
# Image URLs use the /data/local-files/?d=... format so Label Studio can
# serve them through its local-files endpoint from inside Docker.

import json
import math

def obb_corners_to_rotated_rect(corners_norm, img_w, img_h):
    """
    Convert 4 YOLO-OBB corners (normalized 0-1) into Label Studio's
    RectangleLabels format: (x%, y%, width%, height%, rotation_deg).
    The anchor (x, y) is the corner from which rotation starts.
    """
    # De-normalize to pixels
    pts = [(c[0] * img_w, c[1] * img_h) for c in corners_norm]

    # Sort the 4 corners so we always get a consistent ordering:
    # anchor = corner with smallest y; among those, smallest x.
    pts_sorted = sorted(enumerate(pts), key=lambda p: (p[1][1], p[1][0]))
    anchor_idx, anchor = pts_sorted[0]

    # Find the "next" corner clockwise from anchor — the one that's the top-right
    # (i.e. the side that defines the width axis).
    # We test both neighbors of the anchor in the YOLO corner order and pick
    # the one whose direction has the smaller (more positive) angle to the x-axis.
    n_idx = (anchor_idx + 1) % 4
    p_idx = (anchor_idx + 3) % 4
    next_pt = pts[n_idx]
    prev_pt = pts[p_idx]

    ang_next = math.atan2(next_pt[1] - anchor[1], next_pt[0] - anchor[0])
    ang_prev = math.atan2(prev_pt[1] - anchor[1], prev_pt[0] - anchor[0])

    # Pick whichever angle is closer to 0 (i.e., more horizontal) — that's the
    # "width" direction, and its angle is the rotation of the rectangle.
    if abs(ang_next) <= abs(ang_prev):
        width_vec = (next_pt[0] - anchor[0], next_pt[1] - anchor[1])
        height_vec = (prev_pt[0] - anchor[0], prev_pt[1] - anchor[1])
        rotation = math.degrees(ang_next)
    else:
        width_vec = (prev_pt[0] - anchor[0], prev_pt[1] - anchor[1])
        height_vec = (next_pt[0] - anchor[0], next_pt[1] - anchor[1])
        rotation = math.degrees(ang_prev)

    width_px = math.hypot(*width_vec)
    height_px = math.hypot(*height_vec)

    # Label Studio wants rotation in [0, 360) typically
    if rotation < 0:
        rotation += 360

    return {
        "x":        anchor[0] / img_w * 100,
        "y":        anchor[1] / img_h * 100,
        "width":    width_px / img_w * 100,
        "height":   height_px / img_h * 100,
        "rotation": rotation,
    }


def build_ls_tasks(out_root, class_names, url_prefix):
    """Walk images/ + labels/ and build a Label Studio tasks list."""
    tasks = []
    for img_path in sorted((out_root / "images").glob("*.jpg")):
        img = cv.imread(str(img_path))
        if img is None:
            continue
        ih, iw = img.shape[:2]

        lbl_path = out_root / "labels" / (img_path.stem + ".txt")
        ls_results = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 9:
                        continue
                    cls_id = int(parts[0])
                    if not (0 <= cls_id < len(class_names)):
                        continue
                    coords = list(map(float, parts[1:9]))
                    corners = [(coords[2*i], coords[2*i+1]) for i in range(4)]

                    rect = obb_corners_to_rotated_rect(corners, iw, ih)
                    ls_results.append({
                        "original_width": iw,
                        "original_height": ih,
                        "image_rotation": 0,
                        "value": {
                            "x":         rect["x"],
                            "y":         rect["y"],
                            "width":     rect["width"],
                            "height":    rect["height"],
                            "rotation":  rect["rotation"],
                            "rectanglelabels": [class_names[cls_id]],
                        },
                        "from_name": "label",
                        "to_name":   "image",
                        "type":      "rectanglelabels",
                    })

        task = {
            "data": {
                "image": url_prefix + img_path.name,
            },
        }
        if ls_results:
            task["predictions"] = [{
                "model_version": "yolo-obb-auto",
                "result": ls_results,
            }]
        tasks.append(task)
    return tasks


tasks = build_ls_tasks(out_root, CLASS_NAMES, LS_URL_PREFIX)
tasks_path = out_root / "tasks.json"
with open(tasks_path, "w") as f:
    json.dump(tasks, f, indent=2)

# Stats
with_preds = sum(1 for t in tasks if "predictions" in t)
total_boxes = sum(len(t["predictions"][0]["result"])
                  for t in tasks if "predictions" in t)
print(f"Wrote {tasks_path}")
print(f"  Tasks:        {len(tasks)}")
print(f"  With preds:   {with_preds}")
print(f"  Total boxes:  {total_boxes}")
print(f"  Sample URL:   {tasks[0]['data']['image'] if tasks else 'n/a'}")

# Sanity check: print where the files actually ended up
sample_tile = next((out_root / "images").glob("*.jpg"), None)
phys_sample = str(sample_tile) if sample_tile else "<no tiles>"
url_sample = (tasks[0]['data']['image'] if tasks else '<no tasks>')

print(f"""
── FILES ON DISK ──────────────────────────────────────────────────────────────

Images are physically at:
    {out_root / 'images'}
Sample file:
    {phys_sample}

These paths should be inside your mydata/ folder, because mydata is what
Label Studio's Docker container has bind-mounted at /label-studio/data.

── HOW LABEL STUDIO RESOLVES THE URLs ────────────────────────────────────────

Sample URL in tasks.json:
    {url_sample}

With the Docker command:
    docker run -v $(pwd)/mydata:/label-studio/data \\
               -e LOCAL_FILES_SERVING_ENABLED=true \\
               -e LOCAL_FILES_DOCUMENT_ROOT=/label-studio/data \\
               heartexlabs/label-studio

that URL resolves to this file inside the container:
    /label-studio/data/big_tiles/<tile>.jpg

which is this file on your host:
    mydata/big_tiles/<tile>.jpg

── HOW TO IMPORT ──────────────────────────────────────────────────────────────

1. Create a new Label Studio project with this labeling config:

       <View>
         <Image name="image" value="$image"/>
         <RectangleLabels name="label" toName="image" canRotate="true">
           <Label value="chain"         background="#4a6fa5"/>
           <Label value="double"        background="#8b5e83"/>
           <Label value="double treble" background="#6faa6f"/>
           <Label value="enseble_chain" background="#b07a3a"/>
           <Label value="fan"           background="#c45a4a"/>
           <Label value="half_double"   background="#5a8a7a"/>
           <Label value="noise"         background="#999999"/>
           <Label value="single"        background="#c4943a"/>
           <Label value="treble"        background="#5aaa5a"/>
         </RectangleLabels>
       </View>

2. Project Settings → Cloud Storage (optional) — NOT needed; local-files
   serving from the Docker mount is sufficient.

3. Project → Import → upload tasks.json from {out_root}. Each task arrives
   pre-populated with the YOLO detections as editable rotated boxes.

── TROUBLESHOOTING "Could not load URL" ──────────────────────────────────────

- Put the images in mydata/ (already done — this notebook writes there).
- Verify LOCAL_FILES_SERVING_ENABLED=true in the container environment.
- Verify LOCAL_FILES_DOCUMENT_ROOT=/label-studio/data matches the mount
  target (where you bind mydata into the container).
- Test one URL directly in your browser:
      http://localhost:8080/data/local-files/?d=big_tiles/{sample_tile.name if sample_tile else '<tile>.jpg'}
  If that 404s:
    * the Docker mount is wrong (is mydata really mounted to /label-studio/data?)
    * LOCAL_FILES_DOCUMENT_ROOT is set to a different path
    * the file isn't actually inside mydata/ (re-run cell 1 + cell 4)
- If LOCAL_FILES_DOCUMENT_ROOT points at a different parent directory (e.g.
  the project root rather than mydata), edit LS_URL_PREFIX in cell 1 to
  match — for example "/data/local-files/?d=mydata/big_tiles/" — and re-run
  this cell to regenerate tasks.json.
""")


Wrote /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/tasks.json
  Tasks:        55
  With preds:   55
  Total boxes:  10498
  Sample URL:   /data/local-files/?d=test_data/big_tiles/images/11_tile_000.jpg

── FILES ON DISK ──────────────────────────────────────────────────────────────

Images are physically at:
    /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/images
Sample file:
    /Users/elevchenko/Documents/DataScience/Crochet/mydata/test_data/big_tiles/images/11_tile_042.jpg

These paths should be inside your mydata/ folder, because mydata is what
Label Studio's Docker container has bind-mounted at /label-studio/data.

── HOW LABEL STUDIO RESOLVES THE URLs ────────────────────────────────────────

Sample URL in tasks.json:
    /data/local-files/?d=test_data/big_tiles/images/11_tile_000.jpg

With the Docker command:
    docker run -v $(pwd)/mydata:/label-studio/data \
               -e LOCAL_FILES_SERVING_ENABLED=true \
      